In [23]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import DistMult
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
df = pd.read_csv('../data/edges/clean_triples.csv')


In [34]:
model = CustomDistMult(
    num_nodes=len(rawid2id),
    num_relations = len(pred2id),
    hidden_channels=128
).to(device)

In [35]:
dataset = TensorDataset(train_triplets)
dataloader = DataLoader(dataset, batch_size=10000, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in tqdm(range(5)):
    for batch in dataloader:
        h_train = batch[0][:, 0]
        r_train = batch[0][:, 1]
        t_train = batch[0][:, 2]

        optimizer.zero_grad()
        loss = model.loss(h_train, r_train, t_train)
        loss.backward()
        optimizer.step()
    print(loss.item())


h_test = test_triplets[:2000, 0]
r_test = test_triplets[:2000, 1]
t_test = test_triplets[:2000, 2]


model.eval()
with torch.no_grad():
    print(model.test(h_test, r_test, t_test, 8192, filtered_dict=filtered_dict, sampling=False))


 20%|██        | 1/5 [00:22<01:28, 22.05s/it]

0.04755006358027458


 40%|████      | 2/5 [00:44<01:06, 22.15s/it]

0.02945554070174694


 60%|██████    | 3/5 [01:06<00:44, 22.15s/it]

...

0.01858709752559662


100%|██████████| 5/5 [01:50<00:00, 22.15s/it]


0.013495699502527714


100%|██████████| 2000/2000 [01:36<00:00, 20.75it/s]

(2759.22802734375, 0.3644828796386719, {1: '0.3145', 5: '0.4120', 10: '0.4525', 50: '0.5645'})


In [ ]:
(2844.907470703125, 0.13365311920642853, {1: '0.0800', 5: '0.1780', 10: '0.2255', 50: '0.3970'})